# ImageNet100-C EATTA / CAS / OATTA Evaluation Framework

### Mount Google Drive
Connects Colab to your Drive so the repo, dataset, stream CSV, and results folder are accessible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Check Runtime
Prints GPU and PyTorch/CUDA details before running the ImageNet stream jobs.


In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


/bin/bash: line 1: nvidia-smi: command not found
torch: 2.10.0+cpu
cuda available: False


### Set Paths
Defines the Drive repo path, dataset zip, stream CSV, local repo copy, data directory, and output directory.


In [ ]:
REPO_DRIVE_DIR = "/content/drive/MyDrive/DL_project/EATTA+_imagenet"
DATA_ZIP = "/content/drive/MyDrive/DL_project/ImageNet-100-C.zip"
STREAM_CSV = "/content/drive/MyDrive/DL_project/streams2/stream_gradual.csv" # changing for stream type
SAVE_DIR = "/content/drive/MyDrive/DL_project/imagenet_CAT/gradual"

REPO_DIR = "/content/EATTA_imagenet"
DATA_DIR = "/content/ImageNet-100-C"


### Prepare Colab Workspace
Copies the repo from Drive into `/content` and extracts ImageNet-100-C for the run.


In [ ]:
# downloading from git
!rm -rf /content/EATTA_imagenet /content/ImageNet-100-C
!cp -r "{REPO_DRIVE_DIR}" /content/EATTA_imagenet
!unzip -q -o "{DATA_ZIP}" -d /content
!mkdir -p "{SAVE_DIR}"


### Optional Scratch Cell
Inactive helper cell kept for quick Colab path or debug checks.


In [ ]:
# %cd /content
# print("hello")

### Verify Inputs
Checks that the repo, dataset zip, stream CSV, and save directory paths exist.


In [ ]:
import os

print("repo folder exists:", os.path.isdir(REPO_DIR))
print("data zip exists:", os.path.isfile(DATA_ZIP))
print("stream csv exists:", os.path.isfile(STREAM_CSV))
print("save dir exists:", os.path.isdir(SAVE_DIR))

print("test_time.py exists:", os.path.isfile(f"{REPO_DIR}/classification/test_time.py"))
print("source yaml exists:", os.path.isfile(f"{REPO_DIR}/classification/cfgs/imagenet100_c/source.yaml"))
print("eatta yaml exists:", os.path.isfile(f"{REPO_DIR}/classification/cfgs/imagenet100_c/eatta.yaml"))


repo folder exists: True
data zip exists: True
stream csv exists: True
save dir exists: True
test_time.py exists: True
source yaml exists: True
eatta yaml exists: True


### Locate Dataset Folder
Searches `/content` for the extracted `ImageNet-100-C` directory.


In [ ]:
!find /content -maxdepth 3 -type d -name "ImageNet-100-C" -print


/content/ImageNet-100-C
/content/__MACOSX/ImageNet-100-C


### Select Data Directory
Sets `DATA_DIR` to the extracted dataset folder used by `test_time.py`.


In [ ]:
DATA_DIR = "/content/ImageNet-100-C"
print("using DATA_DIR =", DATA_DIR)


using DATA_DIR = /content/ImageNet-100-C


### Install Dependencies
Installs the repo dependencies in Colab, then returns to the classification folder.


In [ ]:
%cd /content/EATTA_imagenet
!pip install -q timm==0.9.16 open-clip-torch==2.24.0 yacs==0.1.8 iopath==0.1.10 webdataset==0.2.86 gdown==5.1.0 ftfy==6.2.0
%cd /content/EATTA_imagenet/classification


/content/EATTA_imagenet
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 4.4 MB/s eta 0:00:00
/content/EATTA_imagenet/classification


### Preview Stream CSV
Validates the sequential stream CSV and prints a few rows for sanity checking.


In [ ]:
import csv

assert os.path.isdir(DATA_DIR), DATA_DIR
assert os.path.isfile(STREAM_CSV), STREAM_CSV

with open(STREAM_CSV, "r") as f:
    reader = csv.DictReader(f)
    first_row = next(reader)
    print("first row:", first_row)


first row: {'position': '0', 'path': '/content/data/ImageNet-100-C/fog/1/n01682714/n01682714_10191.JPEG', 'label': 'n01682714', 'corruption': 'fog', 'severity': '1', 'is_shift': '1'}


### Run Source Baseline
Runs the non-adaptive source model on the same stream and writes source metrics.


In [ ]:
# running source resnet 50 only for imagenet streams
!python test_time.py --cfg cfgs/imagenet100_c/source.yaml \
  DATA_DIR "{DATA_DIR}" \
  CORRUPTION.STREAM_CSV "{STREAM_CSV}" \
  SAVE_DIR "{SAVE_DIR}" \
  TEST.NUM_WORKERS 2 ; echo EXIT_CODE:$?



### Start Log Capture
Redirects notebook stdout/stderr to a timestamped text log on Drive.


In [ ]:
import sys
import logging
from datetime import datetime

# ── Create a timestamped output file on your drive ───────────────────
timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
stream_name = "gradual"   # change per run
log_path   = f"/content/drive/MyDrive/DL_project/results_improv1/{stream_name}_{timestamp}.txt"

# ── Tee class: writes to both stdout and file simultaneously ─────────
class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

import os
os.makedirs(os.path.dirname(log_path), exist_ok=True)
log_file       = open(log_path, 'w')
sys.stdout     = Tee(sys.__stdout__, log_file)
sys.stderr     = Tee(sys.__stderr__, log_file)

# Also capture logger output
file_handler   = logging.FileHandler(log_path)
file_handler.setFormatter(logging.Formatter(
    '[%(asctime)s] [%(filename)s: %(lineno)4d]: %(message)s',
    datefmt='%y/%m/%d %H:%M:%S'
))
logging.getLogger().addHandler(file_handler)

print(f"Logging to: {log_path}")

### Run Vanilla EATTA
Runs EATTA with CAS and OATTA disabled, giving the baseline adaptation result.


In [ ]:
# running vanilla EATTA baseline for imagenet stream - batch size 64
!python test_time.py --cfg cfgs/imagenet100_c/eatta.yaml \
  DATA_DIR "{DATA_DIR}" \
  CORRUPTION.STREAM_CSV "{STREAM_CSV}" \
  SAVE_DIR "{SAVE_DIR}" \
  TEST.NUM_WORKERS 2 \
  CAS.ENABLED False \
  OATTA.ENABLED False ; echo EXIT_CODE:$?



### Run CAS + OATTA
Runs the full implementation with CAS selection and OATTA temporal filtering enabled.


In [ ]:
# running CAS + OATTA implementation for imagenet stream - batch size 64
!python test_time.py --cfg cfgs/imagenet100_c/eatta.yaml \
  DATA_DIR "{DATA_DIR}" \
  CORRUPTION.STREAM_CSV "{STREAM_CSV}" \
  SAVE_DIR "{SAVE_DIR}" \
  TEST.NUM_WORKERS 2 \
  CAS.ENABLED True \
  OATTA.ENABLED True ; echo EXIT_CODE:$?



### Close Log Capture
Restores notebook output and closes the Drive log file.


In [ ]:
log_file.close()
sys.stdout = sys.__stdout__
sys.stderr = sys.__stderr__
print(f"Results saved to: {log_path}")

### List Result Folders
Shows the newest result folders under `SAVE_DIR`.


In [ ]:
!ls -lt "{SAVE_DIR}" | head -20


### Find Recent Runs
Prints recent ImageNet100-C run folders so you can confirm all runs were created.


In [ ]:
import glob
import os

runs = sorted(glob.glob(os.path.join(SAVE_DIR, "*imagenet100_c*")), key=os.path.getmtime)
print("\n".join(runs[-10:]))


### Merge Results and Plot
Loads the latest source, EATTA, and CAS+OATTA JSONs, saves merged metrics, and displays all plots.


In [ ]:
# Display all metrics plots from separate Source, vanilla EATTA, and CAS+OATTA runs
import os
import sys
import json
import glob
from IPython.display import display

# The notebook is usually running from /content/EATTA_imagenet/classification.
# visualise_results.py lives at the repo root.
sys.path.insert(0, REPO_DIR)

from visualise_results import generate_all_plots, merge_separate_run_results


def load_result(path):
    with open(path, "r") as f:
        result = json.load(f)
    if isinstance(result, list):
        result = result[-1]
    result["results_path"] = path
    return result


result_files = sorted(
    glob.glob(os.path.join(SAVE_DIR, "*imagenet100_c_*", "results.json")),
    key=os.path.getmtime,
)

print("Found results.json files:")
for path in result_files[-10:]:
    print(" ", path)

source_runs = []
eatta_runs = []
cas_oatta_runs = []

for path in result_files:
    result = load_result(path)
    variant = result.get("method_variant")
    cas_enabled = result.get("cas_enabled")
    oatta_enabled = result.get("oatta_enabled")
    folder = os.path.basename(os.path.dirname(path))

    if variant == "source" or folder.startswith("source_imagenet100_c_"):
        source_runs.append(result)
    elif variant == "eatta" or (cas_enabled is False and oatta_enabled is False):
        eatta_runs.append(result)
    elif variant == "cas_oatta" or (cas_enabled is True and oatta_enabled is True):
        cas_oatta_runs.append(result)

if not source_runs:
    raise FileNotFoundError(f"No source results.json found under {SAVE_DIR}. Run the source cell first.")
if not eatta_runs:
    raise FileNotFoundError(f"No vanilla EATTA results.json found under {SAVE_DIR}. Run the vanilla EATTA cell first.")
if not cas_oatta_runs:
    raise FileNotFoundError(f"No CAS+OATTA results.json found under {SAVE_DIR}. Run the CAS+OATTA cell first.")

source_results = source_runs[-1]
eatta_results = eatta_runs[-1]
cas_oatta_results = cas_oatta_runs[-1]

print("Using source:", source_results["results_path"])
print("Using vanilla EATTA:", eatta_results["results_path"])
print("Using CAS+OATTA:", cas_oatta_results["results_path"])

results = merge_separate_run_results(source_results, eatta_results, cas_oatta_results)
metrics_log = results.get("metrics_log", {})
stream_config = {
    "stream_name": results.get("stream_name", "imagenet100_c_stream"),
    "corruption_boundaries": metrics_log.get("corruption_boundaries", []),
}

plot_dir = os.path.join(os.path.dirname(cas_oatta_results["results_path"]), "merged_plots")
merged_path = os.path.join(os.path.dirname(cas_oatta_results["results_path"]), "results_merged.json")
merged_latest_path = os.path.join(SAVE_DIR, "results_merged_latest.json")

for path in [merged_path, merged_latest_path]:
    with open(path, "w") as f:
        json.dump(results, f, indent=2)

figs = generate_all_plots(results, stream_config, output_dir=plot_dir)

print("Saved merged results to:", merged_path)
print("Saved latest merged results to:", merged_latest_path)
print("Saved plots to:", plot_dir)
for name, fig in figs.items():
    print("Displaying:", name)
    display(fig)


